# 01 Data Cleaning

**Project:** RetailPulse — E-commerce Revenue & Customer Retention Analytics  
**Dataset:** UCI Online Retail II — ~1.07M invoice line items from a UK online gift retailer, Dec 2009 – Dec 2011

**Goal of this notebook:** load the raw transactions, profile the data quality problems, and produce a clean dataset the rest of the project can rely on.

---

## Step 1 — Load the raw data

The path is *relative* (`../data/raw/...`) rather than absolute, so this notebook runs on any machine straight after cloning the repo.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/online_retail_II.csv")

In [2]:
df.shape

(1067371, 8)

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Step 2 - Profile the raw data

Before cleaning anything, find out what's actually wrong: wrong data types,
missing values, duplicates, and impossible numbers.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


In [5]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(34335)

In [7]:
df[["Quantity", "Price"]].describe().round(2)

,Quantity,Price
count,1067371.00,1067371.00
mean,9.94,4.65
std,172.71,123.55
min,-80995.00,-53594.36
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,80995.00,38970.00


### Data quality findings

| Problem | Scale | Decision |
|---|---|---|
| `Customer ID` missing | 243,007 rows (22.8%) | Keep for revenue; exclude from customer-level analysis |
| `Description` missing | 4,382 rows (0.4%) | No action — overlaps with junk rows removed below |
| Exact duplicate rows | 34,335 (3.2%) | Drop — systems record quantity, not repeated lines |
| Negative `Quantity` (min −80,995) | — | Cancellations/returns. Flag, don't delete |
| Negative `Price` (min −53,594) | — | Accounting adjustments, not products. Filter to `Price > 0` |
| `InvoiceDate` stored as text | all rows | Convert to datetime — needed for cohorts and recency |
| `Customer ID` stored as decimal | all rows | Convert to nullable integer |

**Note on skew:** `Quantity` has mean 9.94 vs median 3, std 172.71 — heavily
right-skewed. Prefer medians in EDA; log-transform before clustering.

## Step 3 - Fix data types

`InvoiceDate` is text and `Customer ID` is a decimal. Both need fixing before
any date-based or customer-based analysis is possible.

In [8]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Customer ID"] = df["Customer ID"].astype("Int64")

In [9]:
df.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                   str
dtype: object

## Step 4 - Remove exact duplicate rows

34,335 rows are exact copies across all 8 columns. A real system records
"2 units" on one line rather than repeating the line, so these are export
artifacts, not genuine sales.

In [10]:
df = df.drop_duplicates()
df.shape

(1033036, 8)

## Step 5 - Flag cancellations

Invoices starting with "C" are cancellations. Flag them rather than deleting
them, so cancelled revenue can be measured later.

In [11]:
df["is_cancelled"] = df["Invoice"].str.startswith("C")
df["is_cancelled"].sum()

np.int64(19104)

In [12]:
pd.crosstab(df["is_cancelled"], df["Quantity"] < 0)

Quantity,False,True
is_cancelled,,
False,1010539,3393
True,1,19103


## Step 6 - Remove non-product rows

Real product codes are 5 digits, optionally followed by letters. Everything
else is postage, bank charges, discounts, test entries or gift vouchers.
Zero and negative prices are accounting adjustments, not sales.

In [13]:
df = df[df["StockCode"].str.match(r"^\d{5}")]
df.shape

(1027056, 9)

In [14]:
df = df[df["Price"] > 0]
df.shape

(1021128, 9)

In [15]:
df["Description"] = df["Description"].str.strip()

In [16]:
((df["Quantity"] < 0) & (~df["is_cancelled"])).sum()

np.int64(0)

In [17]:
df.isna().sum()

Invoice              0
StockCode            0
Description          0
Quantity             0
InvoiceDate          0
Price                0
Customer ID     226965
Country              0
is_cancelled         0
dtype: int64

In [18]:
df[["Quantity", "Price"]].describe().round(2)

,Quantity,Price
count,1021128.00,1021128.00
mean,10.50,3.36
std,169.09,4.87
min,-80995.00,0.03
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.13
max,80995.00,1157.15


### Cleaning summary

| Step | Rows removed | Reason |
|---|---|---|
| Raw load | — | 1,067,371 rows |
| Exact duplicates | 34,335 | Identical across all 8 columns — export artifacts |
| Non-product StockCodes | 5,980 | Postage, bank charges, discounts, tests, vouchers |
| Price <= 0 | 5,928 | Accounting adjustments; also caught all 3,393 warehouse write-offs |
| **Final** | **46,243 (4.3%)** | **1,021,128 rows retained** |

**Known trade-off:** the 5-digit rule also dropped ~160 rows of genuine products
with non-standard codes (`DCGS*`, `SP1002`) — 0.016% of data. Accepted in
exchange for a one-line, maintainable rule.

**Result:** 46,921 invoices · 5,875 customers · 4,892 products · Dec 2009–Dec 2011

## Step 7  Save the clean dataset

Save to parquet rather than CSV so data types survive. Notebooks 02–04 load
this file directly instead of repeating the cleaning.

In [19]:
df.to_parquet("../data/processed/retail_clean.parquet", index=False)

In [20]:
check = pd.read_parquet("../data/processed/retail_clean.parquet")
check.dtypes

Invoice                    str
StockCode                  str
Description                str
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID              Int64
Country                    str
is_cancelled              bool
dtype: object